# BiLSTM Speech Enhancement - Colab training

**Before running:** set the runtime to **GPU** (Runtime -> Change runtime type -> T4 GPU).

One-time dataset prep on your PC (see `scripts/prepare_dataset.md`):
```powershell
cd <repo>
tar -cvf voicebank_demand.tar -C speech clean_trainset_wav noisy_trainset_wav clean_testset_wav noisy_testset_wav
```
then upload `voicebank_demand.tar` (~2.6 GB) to the root of your Google Drive (`MyDrive/`).

In [ ]:
# 1. Clone the rebuild branch (safe to re-run after a runtime reset/reconnect —
# if /content/repo already exists this just updates it instead of failing)
import os
if not os.path.isdir('/content/repo/.git'):
    !git clone --branch bilstm-rebuild https://github.com/AsimShareef/BiLSTM-Speech-Enhancement-System.git /content/repo
else:
    !git -C /content/repo fetch origin bilstm-rebuild
    !git -C /content/repo checkout bilstm-rebuild
    !git -C /content/repo reset --hard origin/bilstm-rebuild
%cd /content/repo
!git log --oneline -3

In [ ]:
%cd /content/repo
# 2. Dependencies (Colab already has tensorflow + numpy/scipy/librosa)
!pip -q install pystoi pesq
import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
%cd /content/repo
# 3. Pull the dataset from Drive and extract to ./speech
# Safe to re-run: drive.mount() on an already-mounted Drive is a no-op, and
# tar -xf just overwrites the same files if speech/ was already extracted.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p speech
!tar -xf /content/drive/MyDrive/voicebank_demand.tar -C speech
!ls speech && echo '---' && ls speech/clean_trainset_wav | wc -l

In [ ]:
%cd /content/repo
# 4. Sanity-check the streaming dataset (no training yet)
!python src/dataset.py

In [ ]:
%cd /content/repo
# 5. Train on the full corpus (early stopping usually halts well before 40 epochs)
# --checkpoint-dir points at Drive: if the Colab runtime disconnects mid-run,
# the best weights / history so far are already saved there, not lost with the VM.
!python src/train.py --epochs 40 --batch 128 \
    --checkpoint-dir /content/drive/MyDrive/bilstm_se_out/checkpoints

In [ ]:
%cd /content/repo
# 6. Evaluate on all 824 test files: BiLSTM vs spectral-subtraction vs noisy
# Reads the model from Drive (written live during training by --checkpoint-dir
# in cell 5), so this works even in a fresh runtime that never ran cell 5 itself.
!python src/evaluate.py --model /content/drive/MyDrive/bilstm_se_out/checkpoints/bilstm_enhancer.keras
print(open('results/metrics.md').read())

In [ ]:
%cd /content/repo
# 7. Save the remaining artefacts back to Drive (model + checkpoints already
# went there live during training via --checkpoint-dir; this just adds results/)
!cp -r results /content/drive/MyDrive/bilstm_se_out/
print('done - MyDrive/bilstm_se_out/ now has: checkpoints/ (model, history.csv, history.json) + results/')
print('download results/metrics.md and checkpoints/history.csv from Drive and share them back')